# Neural candidate generation for Avito services

Этот notebook добавляет к lexical-поиску отдельный dense retrieval канал — по
той же схеме, которую Авито использует для расширения полноты выдачи.

Модель: **`codefuse-ai/ML-Embed-0.6B`**, ICML 2026. Это instruction-aware
multilingual embedder на основе Qwen3-0.6B; русский явно входит в список
поддерживаемых языков. Модель запускается локально в Kaggle, внешних API нет.

Результат notebook:

- честный Recall@50/100/200 на тех же 3 200 validation-группах;
- dense top-200 для validation и benchmark;
- `dense_answer.csv` для диагностики;
- `ml_embed_dense_top200.npz` для последующего fusion с word/char pipeline.

Перед запуском: выбрать GPU accelerator и включить Internet для первой загрузки
модели с Hugging Face.

In [ ]:
# Kaggle обычно уже содержит torch. Обновляем только retrieval-стек.
!pip install -q transformers==4.57.1 sentence-transformers==5.1.2 pyarrow==19.0.1

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import shutil
import subprocess
import time
import zipfile

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

SEED = 20260927
MODEL_ID = 'codefuse-ai/ML-Embed-0.6B'
VALIDATION_GROUPS = 3200
TOP_K = 200
MAX_SEQ_LENGTH = 256
BATCH_SIZE = 64

assert torch.cuda.is_available(), 'Включи GPU accelerator в настройках Kaggle'
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
major, _ = torch.cuda.get_device_capability(0)
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print('GPU:', gpu_name, f'{gpu_gb:.1f} GB', 'dtype:', DTYPE)
print('torch:', torch.__version__)

## 1. Поиск входных файлов

Notebook принимает либо три parquet-файла, либо исходный zip. Название Kaggle
dataset не зашито — можно загрузить архив под любым именем.

In [ ]:
input_root = Path('/kaggle/input')
working = Path('/kaggle/working')

def find_file(name):
    matches = list(input_root.rglob(name))
    return matches[0] if matches else None

train_path = find_file('train.parquet')
queries_path = find_file('benchmark_queries.parquet')
items_path = find_file('benchmark_items.parquet')

if not all([train_path, queries_path, items_path]):
    archives = list(input_root.rglob('*.zip'))
    if not archives:
        # Public archive supplied by the task authors. This keeps the Kaggle
        # kernel self-contained and avoids uploading a duplicate private copy.
        import requests
        public_key = 'https://disk.yandex.ru/d/sNhfo0YOjGtufg'
        response = requests.get(
            'https://cloud-api.yandex.net/v1/disk/public/resources/download',
            params={'public_key': public_key}, timeout=60,
        )
        response.raise_for_status()
        archive_path = working / 'dataset_clean.zip'
        with requests.get(response.json()['href'], stream=True, timeout=120) as download:
            download.raise_for_status()
            total_bytes = int(download.headers.get('content-length', 0))
            with archive_path.open('wb') as output:
                chunks = download.iter_content(chunk_size=8 * 1024 * 1024)
                for chunk in tqdm(
                    chunks,
                    total=(total_bytes + 8 * 1024 * 1024 - 1) // (8 * 1024 * 1024) if total_bytes else None,
                    desc='Downloading task archive', unit='chunk',
                ):
                    if chunk:
                        output.write(chunk)
        archives = [archive_path]
        print('Downloaded public task archive:', archive_path.stat().st_size / 2**20, 'MiB')
    extract_dir = working / 'avito_data'
    extract_dir.mkdir(exist_ok=True)

    # Ссылка задания ведёт на папку Яндекс Диска. Поэтому API отдаёт внешний
    # архив папки, внутри которого лежит ещё один dataset.zip. Распаковываем
    # архивы по слоям, пока не найдём parquet (не глубже трёх уровней).
    pending_archives = [Path(archives[0])]
    unpacked = set()
    for depth in range(3):
        current = [p for p in pending_archives if str(p) not in unpacked]
        if not current:
            break
        for archive_path in current:
            target = extract_dir / f'layer_{depth}_{archive_path.stem}'
            target.mkdir(parents=True, exist_ok=True)
            print(f'Extracting layer {depth}:', archive_path, '->', target)
            with zipfile.ZipFile(archive_path) as archive:
                members = archive.infolist()
                for member in tqdm(members, desc=f'Extracting layer {depth}', unit='file'):
                    archive.extract(member, target)
            unpacked.add(str(archive_path))
        parquet_files = list(extract_dir.rglob('*.parquet'))
        if parquet_files:
            break
        pending_archives = list(extract_dir.rglob('*.zip'))

    def find_extracted(name):
        matches = list(extract_dir.rglob(name))
        if not matches:
            raise FileNotFoundError(
                f'{name} not found after nested archive extraction; '
                f'archives seen: {[str(p) for p in pending_archives]}'
            )
        return matches[0]

    train_path = find_extracted('train.parquet')
    queries_path = find_extracted('benchmark_queries.parquet')
    items_path = find_extracted('benchmark_items.parquet')

print(train_path)
print(queries_path)
print(items_path)

## 2. Тексты и leakage-safe validation

Validation повторяет локальный эксперимент: берём train-клики только по
объявлениям, присутствующим в benchmark-корпусе, группируем по полному контексту
запроса и фиксированно выбираем 3 200 групп. Модель не обучается на этих строках;
это zero-shot проверка dense retrieval.

In [ ]:
query_cols = [
    'search_query', 'search_location_id', 'search_is_delivery_search',
    'search_infm_params_text', 'search_category',
]
train = pd.read_parquet(train_path, columns=query_cols + ['item_id'])
queries = pd.read_parquet(queries_path)
items = pd.read_parquet(items_path)

item_ids = items.item_id.astype(str).to_numpy()
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}
corpus_ids = set(item_ids)

val_rows = train.loc[train.item_id.isin(corpus_ids), query_cols + ['item_id']].copy()
val_rows['_group'] = pd.util.hash_pandas_object(
    val_rows[query_cols].fillna(''), index=False
).astype(str)
relevant = val_rows.groupby('_group').item_id.agg(set)
validation = val_rows.drop_duplicates('_group').set_index('_group')
validation['relevant'] = relevant
validation = validation.sample(min(VALIDATION_GROUPS, len(validation)), random_state=SEED)
validation['relevant_idx'] = validation.relevant.map(
    lambda values: {item_to_idx[value] for value in values}
)

print('train:', train.shape, 'queries:', queries.shape, 'items:', items.shape)
print('validation groups:', len(validation))
print('mean labels/group:', validation.relevant.map(len).mean())
del train, val_rows, relevant
gc.collect()

In [ ]:
def clean(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

def make_query_text(frame):
    main = frame.search_query.fillna('').map(clean)
    params = frame.search_infm_params_text.fillna('').map(clean)
    # Фильтры повторяем один раз, но не добавляем числовые location/category IDs:
    # их наш структурный reranker обрабатывает отдельно.
    return (main + (' | filters: ' + params).where(params.ne(''), '')).tolist()

def make_item_text(frame):
    title = frame.item_title_raw.fillna('').map(clean)
    params = frame.item_infm_params_text.fillna('').map(clean)
    desc = frame.item_description_raw.fillna('').map(clean).str.slice(0, 1800)
    return (
        'title: ' + title +
        (' | parameters: ' + params).where(params.ne(''), '') +
        (' | description: ' + desc).where(desc.ne(''), '')
    ).tolist()

item_texts = make_item_text(items)
val_texts = make_query_text(validation)
benchmark_texts = make_query_text(queries)
all_query_texts = val_texts + benchmark_texts
print('documents:', len(item_texts), 'queries to encode:', len(all_query_texts))
print(item_texts[0][:500])

## 3. ML-Embed inference

Документы кодируются без prompt. Для запросов используем англоязычную
инструкцию: авторы модели рекомендуют такой формат даже в multilingual задачах.
`MAX_SEQ_LENGTH=256` выбран осознанно: для коротких запросов и объявлений услуг
хватает заголовка, параметров и начала описания, а throughput заметно выше.

In [ ]:
started = time.time()
model = SentenceTransformer(
    MODEL_ID,
    device='cuda',
    model_kwargs={'torch_dtype': DTYPE},
)
model.max_seq_length = MAX_SEQ_LENGTH

query_prompt = (
    'Instruct: Given a Russian user search query for services on a classifieds '
    'marketplace, retrieve advertisements that satisfy the service intent and '
    'textual filters.\nQuery: '
)

item_embeddings = model.encode_document(
    item_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float16)

query_embeddings = model.encode(
    all_query_texts,
    prompt=query_prompt,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float16)

print('item embeddings:', item_embeddings.shape, item_embeddings.nbytes / 2**20, 'MiB')
print('query embeddings:', query_embeddings.shape)
print('encoding minutes:', (time.time() - started) / 60)

## 4. Exact dense top-k на GPU

Корпус небольшой для brute-force cosine search: 189k × 1024 float16 занимает
около 370 MiB GPU. Это даёт точный top-200 без приближённого ANN и без потери
recall из-за настроек индекса.

In [ ]:
del model, item_texts, all_query_texts
gc.collect()
torch.cuda.empty_cache()

item_tensor = torch.from_numpy(item_embeddings).to('cuda')
all_indices, all_scores = [], []
SEARCH_BATCH = 64
with torch.inference_mode():
    search_starts = range(0, len(query_embeddings), SEARCH_BATCH)
    for start in tqdm(search_starts, desc='Exact cosine top-200', unit='batch'):
        q = torch.from_numpy(query_embeddings[start:start + SEARCH_BATCH]).to('cuda')
        scores = q @ item_tensor.T
        values, indices = torch.topk(scores, k=TOP_K, dim=1)
        all_indices.append(indices.cpu().numpy().astype(np.int32))
        all_scores.append(values.cpu().numpy().astype(np.float16))

top_indices = np.vstack(all_indices)
top_scores = np.vstack(all_scores)
val_indices = top_indices[:len(validation)]
val_scores = top_scores[:len(validation)]
benchmark_indices = top_indices[len(validation):]
benchmark_scores = top_scores[len(validation):]
print(top_indices.shape, top_scores.shape)

## 5. Zero-shot dense Recall

Считаем именно macro Recall@k из условия: сначала recall для каждого полного
контекста запроса, затем среднее. Dense standalone не обязан победить гибрид;
главный вопрос — приносит ли он новые релевантные объекты в candidate union.

In [ ]:
def macro_recall_at_k(rankings, labels, k):
    values = []
    hits = total = 0
    for ranking, relevant_set in zip(rankings, labels):
        found = len(set(map(int, ranking[:k])) & relevant_set)
        values.append(found / len(relevant_set))
        hits += found
        total += len(relevant_set)
    return float(np.mean(values)), hits / total

metrics = {}
labels = validation.relevant_idx.tolist()
for k in (50, 100, 200):
    macro, micro = macro_recall_at_k(val_indices, labels, k)
    metrics[f'recall@{k}'] = {'macro': macro, 'micro': micro}
    print(f'Recall@{k}: macro={macro:.6f}, micro={micro:.6f}')

## 6. Артефакты для fusion

NPZ хранит позиции объявлений в исходном `benchmark_items.parquet`, scores и
идентификаторы групп. Его размер мал: сами эмбеддинги скачивать не требуется.
После этого локально добавляем dense rank/cosine в уже проверенный reranker и
сравниваем с v3 на том же holdout.

In [ ]:
output_npz = Path('/kaggle/working/ml_embed_dense_top200.npz')
np.savez_compressed(
    output_npz,
    model_id=np.asarray([MODEL_ID]),
    val_group_ids=validation.index.astype(str).to_numpy(),
    val_indices=val_indices,
    val_scores=val_scores,
    benchmark_query_ids=queries.query_id.astype(str).to_numpy(),
    benchmark_indices=benchmark_indices,
    benchmark_scores=benchmark_scores,
    item_ids=item_ids,
)

dense_answer = pd.DataFrame({
    'query_id': queries.query_id.astype(str),
    'answer': [' '.join(item_ids[row[:50]]) for row in benchmark_indices],
})
dense_answer.to_csv('/kaggle/working/dense_answer.csv', index=False)

report = {
    'model': MODEL_ID,
    'gpu': gpu_name,
    'dtype': str(DTYPE),
    'max_seq_length': MAX_SEQ_LENGTH,
    'embedding_shape': list(item_embeddings.shape),
    'validation_groups': len(validation),
    'metrics': metrics,
    'runtime_minutes': (time.time() - started) / 60,
}
Path('/kaggle/working/neural_report.json').write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8'
)

# Final format sanity check for diagnostic dense-only submission.
assert dense_answer.shape == (len(queries), 2)
assert dense_answer.query_id.is_unique
assert dense_answer.answer.str.split().str.len().eq(50).all()
print(json.dumps(report, ensure_ascii=False, indent=2))
print('Wrote:', output_npz, Path('/kaggle/working/dense_answer.csv'))

# Не публикуем исходные данные как Kaggle Output: оставляем только компактные
# результаты retrieval. Исходники при следующем запуске снова скачаются.
temporary_data = Path('/kaggle/working/avito_data')
if temporary_data.exists():
    shutil.rmtree(temporary_data)
downloaded_archive = Path('/kaggle/working/dataset_clean.zip')
if downloaded_archive.exists():
    downloaded_archive.unlink()
print('Temporary dataset removed; output contains only fusion artifacts.')

## Что делать после завершения

Скачать из Output:

1. `ml_embed_dense_top200.npz` — главный файл;
2. `neural_report.json` — метрики и конфигурация;
3. `dense_answer.csv` — только диагностика, **не отправлять до fusion**.

Dense-only выдача теряет точные названия и фильтры. Финальная отправка должна
объединять neural candidates с word/char/history/location pipeline.